# v11: Evidence Bank + Candidate Span + Concrete-First Rescue

v11 is derived from v10, but changes the optimization target from conservative refusal to evidence-driven concrete answering. It keeps the byte-budget guard from v10 and adds structured question parsing, an evidence bank, candidate span extraction, ranked evidence packets, and an evidence-span fallback for cases where all model paths return `Unable to determine`.


## 1. 配置与导入

v11 继续复用现有 BM25 和 v3 工具注册，不修改 `agent/tools.py`。差异在 controller 层：扩大默认窗口，并基于 evidence snippet 的 `start` offset 追加读取非开头窗口。


In [1]:
from pathlib import Path
import json
import re
import sys
from typing import Any, Dict, List, Optional

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.dataset_utils import load_jsonl
from agent.eval import run_evaluation
from agent.tools import build_searcher, get_v3_research_tool_specs_and_registry, make_initial_search_queries
from agent.vllm_client import VLLMClient

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"
API_KEY = "dummy"

BM25_INDEX_PATH = str(project_root / "indexes" / "browsecomp_plus_bm25.sqlite")
HARD50_PATH = str(project_root / "browsecomp_plus_hard50.jsonl")
SUBMISSION_PATH = str(project_root / "runs" / "v11_submission.jsonl")
EVAL_OUTPUT_PATH = str(project_root / "runs" / "v11_eval_results.jsonl")
TRACE_DIR = project_root / "trace"
TRACE_PATH = str(TRACE_DIR / "v11_trace.jsonl")
OPEN_TRACK_TRACE_PATH = str(TRACE_DIR / "v11_open_track_trace.jsonl")

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=BM25_INDEX_PATH)
tool_specs, tool_registry = get_v3_research_tool_specs_and_registry(
    searcher=searcher,
    k=8,
    snippet_max_chars=1600,
    document_window_chars=3200,
)


## 2. Prompt

v11 沿用 v9 的保守回答与高阈值验证 prompt。主要优化点在工具读取覆盖率，而不是继续放宽 verifier。


In [2]:
COMPACT_ANSWER_PROMPT = """
You are an evidence-grounded BrowseComp-Plus QA agent.
Use only the retrieved local corpus evidence in the evidence packet.

The evidence packet may contain:
- parsed_question: expected answer type and hard constraints
- top_evidence: ranked snippets/windows from local documents
- candidate_spans: answer candidates extracted from evidence

Rules:
1. Prefer a concrete exact answer when a candidate span is present in evidence and matches the expected answer type.
2. Do not answer from memory; choose from evidence or cite evidence text that directly contains the answer span.
3. If evidence is incomplete but one candidate is clearly best and not contradicted, output that candidate with moderate confidence.
4. Return 'Unable to determine' only when no usable candidate span exists or the best candidates are clearly contradicted.
5. The exact answer must be a clean span, not a citation, bullet, document id, or evidence sentence.

Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()

PLANNER_PROMPT = """
Generate a compact BM25 search expansion plan. Do not answer the question.
Return JSON only with keys search_queries, key_phrases, keywords, answer_type.
Queries should be short, literal, and evidence-seeking. Include phrases that identify the final requested answer type.
""".strip()

DECOMPOSER_PROMPT = """
You are the planner agent in a multi-agent BrowseComp-Plus system.
Decompose the question into independently searchable evidence requirements.
Do not answer the question and do not use outside knowledge.
Return JSON only with keys:
- subquestions: short literal subquestions or clue checks
- constraints: dates, titles, roles, organizations, page ranges, quoted phrases, or other hard clues
- key_entities: names, places, organizations, works, or distinctive noun phrases
- expected_answer_type: one of person, title, date or year, percentage or number, organization, place, short answer
- search_focus: 2-4 concise BM25 query ideas that combine distinctive clues
""".strip()

ADJUDICATOR_PROMPT = """
You choose the best final answer for BrowseComp-Plus using only evidence summaries and candidate outputs.

Rules:
1. Prefer a concrete answer if it is directly present in evidence and has the right answer type.
2. If a concrete answer is only partially supported but no better candidate exists, keep it with lower confidence instead of defaulting to Unable.
3. If candidates conflict and none is supported by evidence, choose 'Unable to determine'.
4. Do not invent a third answer unless it is explicitly stated in evidence.
5. Output the required final answer format.

Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()

ANSWER_VERIFIER_PROMPT = """
You are the verifier agent for BrowseComp-Plus.
Use only the provided evidence packet and candidate answer.
Do not solve from memory.

Score whether the candidate answer is supported by evidence and whether it satisfies the final requested answer type.
Do not reject a candidate just because some background clue is missing if the answer span itself is directly present and there is no contradiction.
Return JSON only with keys:
- support_level: supported, partial, unsupported, contradicted, or unclear
- supported: true or false
- answer: the clean exact answer span if supported or partially supported, otherwise Unable to determine
- confidence: integer 0-100
- evidence: one short evidence summary
- missing_constraints: list of important unsatisfied clues
""".strip()

OPEN_TRACK_FINALIZER_PROMPT = """
You are the finalizer agent in a planner-searcher-verifier BrowseComp-Plus system.
Use only the retrieved local corpus evidence packet.

Rules:
1. Select the best exact answer from candidate_spans when one matches the expected answer type and appears in evidence.
2. Prefer a concrete answer over Unable unless all candidates are contradicted or clearly wrong.
3. If previous concrete candidates failed verification, choose another candidate only if evidence supports it.
4. The exact answer must be a clean span, not a document id, citation prefix, bullet, or full evidence sentence.
5. Do not use memory and do not infer beyond retrieved evidence.

Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()


## 3. 通用辅助函数

v11 继续保留 citation fragment salvage 和 malformed 过滤；新增的文档覆盖逻辑放在工具状态层。


In [3]:
STOPWORDS = {
    "the", "and", "for", "that", "with", "from", "this", "what", "which", "who", "whose", "where",
    "when", "about", "into", "between", "after", "before", "during", "their", "there", "would",
    "could", "should", "please", "provide", "identify", "looking", "question", "answer", "according",
    "want", "know", "something", "simplicity", "refer", "them", "find", "documents", "matching", "clues",
}


def canonical_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "").strip().lower())


def truncate_text(text: str, max_chars: int) -> str:
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "... [truncated]"


def strip_thinking(text: str) -> str:
    text = str(text or "")
    text = re.sub(r"(?is)<think>.*?</think>\s*", "", text).strip()
    text = re.sub(r"(?is)<think>.*", "", text).strip()
    return text


BAD_ANSWER_PREFIXES = (
    "explanation:", "evidence:", "confidence:", "based on", "the evidence", "from the evidence",
    "i cannot", "i can't", "there is not enough", "not enough evidence", "document ", "doc ",
    "citation", "source", "snippet", "quote:", "quoted evidence",
)

MALFORMED_ANSWER_PATTERNS = (
    r"^[-*•]\s*",
    r"(?i)^document\s+\d+\s*:",
    r"(?i)\bdocument\s+\d+\b",
    r"(?i)\bdocid\s*[:#]?\s*\d+\b",
    r"(?i)^quoted?\s+fragment\b",
)


def salvage_document_fragment_answer(raw_answer: str) -> str:
    text = re.sub(r"\s+", " ", str(raw_answer or "")).strip()
    match = re.search(r'(?i)document\s+\d+\s*:\s*["“]?([^"(;\n]{3,90})', text)
    if not match:
        return ""
    span = match.group(1)
    span = re.split(r"(?i)\b(?:annual report|form\s+10-k|quarterly report|report)\b", span)[0]
    span = re.sub(r"(?i)\b(?:incorporated|inc\.?|corp\.?|corporation|ltd\.?|limited|llc|plc)\b\.?", "", span)
    span = re.sub(r"\s+", " ", span).strip(" -–—:;,.\"'")
    if not span or len(span.split()) > 8:
        return ""
    return span


def is_malformed_answer(answer: str) -> bool:
    answer = str(answer or "").strip()
    if not answer:
        return True
    lower = answer.lower()
    if any(lower.startswith(prefix) for prefix in BAD_ANSWER_PREFIXES):
        return True
    if any(re.search(pattern, answer) for pattern in MALFORMED_ANSWER_PATTERNS):
        return True
    if answer.count('"') == 1 or answer.count("'") == 1:
        return True
    if len(answer) > 180 and len(answer.split()) > 24:
        return True
    if re.search(r"(?i)\b(could not|would not|was not|were not|had been|according to)\b", answer) and len(answer.split()) > 6:
        return True
    return False


def clean_answer_value(value: str) -> str:
    answer = strip_thinking(value)
    answer = re.split(r"(?im)^\s*(?:Explanation|Evidence|Confidence)\s*:", answer)[0].strip()
    raw_answer = re.sub(r"\s+", " ", answer).strip()
    salvaged = salvage_document_fragment_answer(raw_answer)
    if salvaged:
        return salvaged
    if is_malformed_answer(raw_answer):
        return "Unable to determine"
    answer = raw_answer.strip(" \t\n\r\"'`*_.,;:")
    if not answer:
        return "Unable to determine"
    lower = answer.lower()
    if any(lower.startswith(prefix) for prefix in BAD_ANSWER_PREFIXES):
        return "Unable to determine"
    if is_malformed_answer(answer):
        return "Unable to determine"
    return answer


def extract_exact_answer(text: str) -> str:
    original = str(text or "").strip()
    clean = strip_thinking(original)
    for candidate_text in (clean, original):
        match = re.search(r"(?im)^\s*Exact Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return clean_answer_value(match.group(1))
        match = re.search(r"(?im)^\s*(?:Final\s+)?Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return clean_answer_value(match.group(1))
    lines = [line.strip() for line in clean.splitlines() if line.strip()]
    for line in reversed(lines):
        if re.match(r"(?i)^(explanation|evidence|confidence)\s*:", line):
            continue
        if 0 < len(line) <= 120:
            return clean_answer_value(line)
    return "Unable to determine"


def extract_confidence(text: str, default: int = 0) -> int:
    match = re.search(r"(?im)^\s*Confidence\s*:\s*(\d{1,3})\s*%", str(text or ""))
    if not match:
        return default
    return max(0, min(100, int(match.group(1))))


def coerce_confidence(value: Any, default: int = 0) -> int:
    try:
        return max(0, min(100, int(float(value))))
    except (TypeError, ValueError):
        return default


def is_unable_answer(answer: str) -> bool:
    return bool(re.search(r"(?i)unable|cannot|can't|not enough|insufficient|not determine|not specified|unknown", str(answer or "")))


def infer_expected_answer_type(question: str) -> str:
    q = canonical_text(question)
    if re.search(r"\b(percent|percentage|rate|ratio|decrease|increase)\b", q):
        return "percentage or number"
    if re.search(r"\b(when|date|year|month|day)\b", q):
        return "date or year"
    if re.search(r"\b(title|chapter|book|novel|song|album|film|movie|report|software|paper|article)\b", q):
        return "title"
    if re.search(r"\b(company|organization|organisation|university|ministry|agency|team|club|publisher)\b", q):
        return "organization"
    if re.search(r"\b(city|country|street|place|where|nationality)\b", q):
        return "place"
    if re.search(r"\b(who|person|name|author|designer|co-author|founder|director|actor|artist|professor)\b", q):
        return "person"
    return "short answer"


def should_expand(candidate: Dict[str, Any], min_confidence: int = 55) -> bool:
    answer = candidate.get("predicted_answer", "")
    confidence = int(candidate.get("confidence", 0) or 0)
    if is_unable_answer(answer):
        return True
    if is_malformed_answer(answer):
        return True
    if confidence and confidence < min_confidence:
        return True
    if len(str(answer).strip()) == 0 or len(str(answer)) > 180:
        return True
    return False


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = strip_thinking(text)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None


def normalize_string_list(value: Any, limit: int = 10, max_chars: int = 160) -> List[str]:
    if isinstance(value, str):
        items = [value]
    elif isinstance(value, list):
        items = value
    else:
        items = []
    output = []
    seen = set()
    for item in items:
        text = " ".join(str(item or "").replace("\n", " ").split()).strip(" ,.;:")
        if len(text) > max_chars:
            text = text[:max_chars].rstrip()
        key = text.lower()
        if text and key not in seen:
            output.append(text)
            seen.add(key)
        if len(output) >= limit:
            break
    return output


def tokenize_for_score(text: str) -> List[str]:
    tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9_\-']+", str(text or "").lower())
    return [token.strip("'-") for token in tokens if len(token.strip("'-")) >= 4 and token not in STOPWORDS]


def rank_search_results(results: List[Dict[str, Any]], terms: List[str], limit: int = 8) -> List[Dict[str, Any]]:
    ranked = []
    seen_docids = set()
    for item in results:
        docid = str(item.get("docid", ""))
        if docid and docid in seen_docids:
            continue
        if docid:
            seen_docids.add(docid)
        haystack = canonical_text(" ".join([item.get("query", ""), item.get("snippet", ""), item.get("url", "")]))
        overlap = sum(1 for term in terms if term in haystack)
        score = float(item.get("score", 0) or 0)
        ranked.append({**item, "controller_score": overlap * 1000 + score, "term_overlap": overlap})
    ranked.sort(key=lambda row: (row.get("term_overlap", 0), row.get("score", 0)), reverse=True)
    return ranked[:limit]


def compact_result_summary(results: List[Dict[str, Any]], max_items: int = 8) -> List[Dict[str, Any]]:
    compact = []
    for item in results[:max_items]:
        compact.append(
            {
                "docid": item.get("docid", ""),
                "score": item.get("score", 0),
                "controller_score": item.get("controller_score", 0),
                "url": item.get("url", ""),
                "query": item.get("query", ""),
                "snippet": truncate_text(item.get("snippet", ""), 450),
            }
        )
    return compact


def qwen_extra_payload(model_name: str) -> Optional[Dict[str, Any]]:
    if "qwen" in str(model_name or "").lower():
        return {"chat_template_kwargs": {"enable_thinking": False}}
    return None


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = strip_thinking(text)
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    candidates = [fenced.group(1)] if fenced else []
    candidates.append(text)
    decoder = json.JSONDecoder()
    for candidate in candidates:
        for match in re.finditer(r"\{", candidate):
            try:
                value, _ = decoder.raw_decode(candidate[match.start():])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
    return None


def final_question_clause(question: str) -> str:
    text = " ".join(str(question or "").split())
    if not text:
        return ""
    markers = [
        "can you tell me",
        "what is",
        "what was",
        "who is",
        "who was",
        "which",
        "where",
        "when",
        "how many",
        "how much",
    ]
    lower = text.lower()
    positions = [lower.rfind(marker) for marker in markers if lower.rfind(marker) >= 0]
    if positions:
        return text[max(positions):]
    pieces = re.split(r"[?]", text)
    return pieces[-2] if len(pieces) > 1 and pieces[-2].strip() else text[-260:]


def parse_question_requirements(question: str) -> Dict[str, Any]:
    text = " ".join(str(question or "").split())
    lower = text.lower()
    final_ask = final_question_clause(text)
    final_lower = final_ask.lower()
    quoted = re.findall(r"[\"'“”‘’]([^\"'“”‘’]{3,120})[\"'“”‘’]", text)
    years = re.findall(r"\b(?:1[5-9]\d{2}|20\d{2})\b", text)
    page_ranges = re.findall(r"\bpages?\s+\d+\s*(?:-|to|–|—)\s*\d+\b|\bpage\s+\d+\b", lower)
    percentages = re.findall(r"\b\d+(?:\.\d+)?\s?%", text)
    numbers = re.findall(r"\b\d+(?:\.\d+)?\b", text)
    terms = tokenize_for_score(text)
    long_terms = [term for term in terms if len(term) >= 5]

    expected = "short answer"
    if re.search(r"\b(percent|percentage|%)\b", final_lower):
        expected = "percentage or number"
    elif re.search(r"\b(how many|how much|number of|total|size|height|length|count)\b", final_lower):
        expected = "percentage or number"
    elif re.search(r"\b(first and last name|full name|name of (?:the )?(?:woman|man|person|author|writer|historian|professor|adviser|advisor|player|artist|founder)|who (?:is|was))\b", final_lower):
        expected = "person"
    elif re.search(r"\b(title of|chapter|book title|article title|paper title|song|album|film|movie|novel|dissertation|work)\b", final_lower):
        expected = "title"
    elif re.search(r"\b(company|organization|organisation|corporation|publisher|university|ministry|agency|firm)\b", final_lower):
        expected = "organization"
    elif re.search(r"\b(where|place|city|country|state|province|located)\b", final_lower):
        expected = "place"
    elif re.search(r"\b(when|date|year)\b", final_lower):
        expected = "date or year"
    elif re.search(r"\b(first and last name|full name)\b", lower):
        expected = "person"
    elif re.search(r"\b(title of|chapter title)\b", lower):
        expected = "title"

    answer_type_terms = {
        "person": ["name", "person", "author", "advisor", "adviser", "professor", "partner"],
        "title": ["title", "chapter", "book", "article", "paper", "novel", "work"],
        "organization": ["company", "organization", "publisher", "university", "ministry", "corporation"],
        "place": ["place", "city", "country", "state", "province"],
        "percentage or number": ["percentage", "percent", "number", "height", "size", "total"],
        "date or year": ["date", "year", "when"],
    }.get(expected, ["answer"])

    hard_constraints = normalize_string_list(
        quoted + page_ranges + years + percentages + long_terms,
        limit=20,
        max_chars=90,
    )
    return {
        "expected_answer_type": expected,
        "final_ask": final_ask,
        "quoted_phrases": normalize_string_list(quoted, limit=8, max_chars=120),
        "years": normalize_string_list(years, limit=10, max_chars=10),
        "page_ranges": normalize_string_list(page_ranges, limit=8, max_chars=40),
        "percentages": normalize_string_list(percentages, limit=8, max_chars=20),
        "numbers": normalize_string_list(numbers, limit=14, max_chars=20),
        "distinctive_terms": normalize_string_list(long_terms, limit=18, max_chars=70),
        "hard_constraints": hard_constraints,
        "answer_type_terms": answer_type_terms,
    }


def build_query_pack(question: str, parsed: Optional[Dict[str, Any]] = None, max_queries: int = 8, max_query_chars: int = 180) -> List[str]:
    parsed = parsed or parse_question_requirements(question)
    queries: List[str] = []

    def add(query: str) -> None:
        query = " ".join(str(query or "").split()).strip(" ,.;:")
        if not query:
            return
        if len(query) > max_query_chars:
            query = query[:max_query_chars].rstrip()
        key = query.lower()
        if key not in {item.lower() for item in queries}:
            queries.append(query)

    for query in make_initial_search_queries(question, max_queries=4, max_query_chars=max_query_chars):
        add(query)
    quoted = parsed.get("quoted_phrases", [])
    years = parsed.get("years", [])
    pages = parsed.get("page_ranges", [])
    distinctive = parsed.get("distinctive_terms", [])
    hard = parsed.get("hard_constraints", [])
    answer_terms = parsed.get("answer_type_terms", [])
    if quoted:
        add(" ".join(quoted[:3] + years[:4]))
    if pages:
        add(" ".join(pages[:3] + distinctive[:8]))
    if hard:
        add(" ".join(hard[:10]))
    if answer_terms:
        add(" ".join(answer_terms[:4] + hard[:8]))
    for index in range(0, min(len(hard), 8), 2):
        add(" ".join(hard[index:index + 2] + years[:3] + answer_terms[:2]))
    return queries[:max_queries]


## 4. 工具执行与轨迹状态

v11 在 `state` 中记录 `opened_windows`，不再只记录 docid。这样同一文档可以在不同 offset 附近打开多个窗口，用来缓解“只读文档开头”的问题。


In [4]:
def normalize_tool_result(tool_name: str, result: Any, max_document_chars: int = 4800) -> Any:
    if tool_name in {"get_document", "get_document_window"} and isinstance(result, dict):
        normalized = dict(result)
        if "text" in normalized:
            full_text = str(normalized.get("text", ""))
            normalized["text"] = truncate_text(full_text, max_document_chars)
            normalized["text_length"] = len(full_text)
            normalized["text_is_truncated"] = len(full_text) > max_document_chars
        return normalized
    return result


def execute_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name not in tool_registry:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": "unknown tool"}
    try:
        raw_result = tool_registry[tool_name](**arguments)
        return {
            "ok": True,
            "tool_name": tool_name,
            "arguments": arguments,
            "tool_result": normalize_tool_result(tool_name, raw_result),
        }
    except Exception as exc:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": repr(exc)}


def make_tool_message(tool_call_id: str, executed: Dict[str, Any]) -> Dict[str, Any]:
    content = executed.get("tool_result") if executed.get("ok") else executed
    return {"role": "tool", "tool_call_id": tool_call_id, "content": json.dumps(content, ensure_ascii=False)}


def docids_from_tool_result(tool_name: str, result: Any) -> List[str]:
    if tool_name in {"search", "search_many"} and isinstance(result, list):
        return [str(item.get("docid", "")) for item in result if isinstance(item, dict) and item.get("docid")]
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return [str(item.get("docid", "")) for item in result.get("snippets", []) if item.get("docid")]
    if isinstance(result, dict) and result.get("docid"):
        return [str(result.get("docid"))]
    return []


def compact_payload(value: Any, max_chars: int = 2200) -> Any:
    try:
        text = json.dumps(value, ensure_ascii=False, default=str)
    except TypeError:
        text = str(value)
    if len(text) <= max_chars:
        return value
    return {"_truncated_json": truncate_text(text, max_chars)}


def record_open_track_event(state: Dict[str, Any], component: str, action: str, payload: Dict[str, Any]) -> None:
    state.setdefault("open_track_trace", []).append(
        {
            "step": len(state.get("open_track_trace", [])) + 1,
            "component": component,
            "action": action,
            "payload": compact_payload(payload),
        }
    )



def initial_state(question: str) -> Dict[str, Any]:
    parsed = parse_question_requirements(question)
    return {
        "question": question,
        "parsed_question": parsed,
        "mode": "compact",
        "search_queries": [],
        "seen_docids": [],
        "opened_docids": [],
        "opened_windows": [],
        "ranked_results": [],
        "evidence_table": [],
        "evidence_bank": [],
        "candidate_span_table": [],
        "tool_events": [],
        "candidate_answers": [],
        "focused_rescue_plan": {},
        "decomposition": {},
        "verification": {},
        "open_track_search": {},
        "open_track_trace": [],
        "decision": {},
    }


def summarize_tool_result(tool_name: str, result: Any) -> str:
    if tool_name in {"search", "search_many"}:
        return "docids=" + ", ".join(docids_from_tool_result(tool_name, result)[:12])
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return f"collected {len(result.get('snippets', []))} snippets"
    if isinstance(result, dict) and result.get("docid"):
        return f"inspected docid={result.get('docid', '')}"
    return truncate_text(json.dumps(result, ensure_ascii=False), 300)



def update_state_from_execution(state: Dict[str, Any], executed: Dict[str, Any], round_id: int) -> None:
    tool_name = executed.get("tool_name", "")
    arguments = executed.get("arguments", {}) or {}
    result = executed.get("tool_result")
    if tool_name == "search_many":
        for query in arguments.get("queries", []) or []:
            if query and query not in state["search_queries"]:
                state["search_queries"].append(query)
    for docid in docids_from_tool_result(tool_name, result):
        if docid and docid not in state["seen_docids"]:
            state["seen_docids"].append(docid)
        if tool_name in {"get_document", "get_document_window", "find_in_document"} and docid not in state["opened_docids"]:
            state["opened_docids"].append(docid)
    if tool_name == "get_document_window" and isinstance(result, dict):
        window_record = {
            "docid": str(result.get("docid", "")),
            "start": int(result.get("start", 0) or 0),
            "end": int(result.get("end", 0) or 0),
            "text_length": int(result.get("text_length", 0) or 0),
        }
        if window_record["docid"] and window_record not in state.setdefault("opened_windows", []):
            state["opened_windows"].append(window_record)
    if tool_name in {"find_in_document", "collect_evidence_snippets"}:
        state["evidence_table"].append(
            {"round_id": round_id, "tool_name": tool_name, "arguments": arguments, "summary": summarize_tool_result(tool_name, result)}
        )
    update_evidence_bank_from_execution(state, executed, round_id)
    state["tool_events"].append(
        {"round_id": round_id, "tool_name": tool_name, "arguments": arguments, "ok": executed.get("ok", False), "summary": summarize_tool_result(tool_name, result)}
    )


def append_controller_tool_call(messages: List[Dict[str, Any]], state: Dict[str, Any], tool_name: str, arguments: Dict[str, Any], round_id: int, note: str) -> Dict[str, Any]:
    call_id = f"controller_{round_id}_{len(state['tool_events']) + 1}_{tool_name}"
    messages.append(
        {
            "role": "assistant",
            "content": note,
            "tool_calls": [
                {"id": call_id, "type": "function", "function": {"name": tool_name, "arguments": json.dumps(arguments, ensure_ascii=False)}}
            ],
        }
    )
    executed = execute_tool(tool_name, arguments)
    messages.append(make_tool_message(call_id, executed))
    update_state_from_execution(state, executed, round_id)
    return executed


def window_already_opened(state: Dict[str, Any], docid: str, start: int, tolerance: int = 900) -> bool:
    for window in state.get("opened_windows", []):
        if str(window.get("docid", "")) != str(docid):
            continue
        if abs(int(window.get("start", 0) or 0) - int(start or 0)) <= tolerance:
            return True
    return False



def open_context_windows_from_snippets(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    snippet_result: Any,
    round_id: int,
    max_windows: int = 2,
    window_chars: int = 3600,
    note: str = "v11 evidence coverage: inspect keyword-hit context window.",
) -> List[Dict[str, Any]]:
    if not isinstance(snippet_result, dict):
        return []
    opened = []
    seen = set()
    snippets = snippet_result.get("snippets", []) or []
    snippets = [item for item in snippets if item.get("docid") and item.get("start") is not None]
    snippets.sort(key=lambda item: score_snippet_for_opening(item, state), reverse=True)
    for item in snippets:
        docid = str(item.get("docid", ""))
        start = max(0, int(item.get("start", 0) or 0) - window_chars // 3)
        key = (docid, start // 900)
        if key in seen or window_already_opened(state, docid, start):
            continue
        seen.add(key)
        executed = append_controller_tool_call(
            messages,
            state,
            "get_document_window",
            {"docid": docid, "start": start, "window_chars": window_chars},
            round_id=round_id,
            note=note,
        )
        opened.append(executed)
        if len(opened) >= max_windows:
            break
    refresh_candidate_span_table(state)
    return opened


MODEL_INPUT_BYTE_BUDGET = 26000
MAX_STATE_MESSAGE_BYTES = 8000
MAX_TOOL_SNIPPET_BYTES = 900
MAX_TOOL_RESULT_BYTES = 1800
MAX_ASSISTANT_CONTEXT_BYTES = 800
MAX_MODEL_MESSAGES = 28


def truncate_to_bytes(text: str, max_bytes: int) -> str:
    text = str(text or "")
    encoded = text.encode("utf-8")
    if len(encoded) <= max_bytes:
        return text
    suffix = "... [byte-truncated]"
    suffix_bytes = suffix.encode("utf-8")
    keep = max(0, int(max_bytes) - len(suffix_bytes))
    return encoded[:keep].decode("utf-8", errors="ignore") + suffix


def compact_tool_payload_for_prompt(value: Any) -> Any:
    if isinstance(value, list):
        return [compact_tool_payload_for_prompt(item) for item in value[:8]]
    if not isinstance(value, dict):
        if isinstance(value, str):
            return truncate_to_bytes(value, MAX_TOOL_RESULT_BYTES)
        return value
    compact: Dict[str, Any] = {}
    for key, item in value.items():
        if key in {"snippet", "text"}:
            compact[key] = truncate_to_bytes(str(item or ""), MAX_TOOL_SNIPPET_BYTES)
        elif key in {"snippets", "matches"} and isinstance(item, list):
            compact[key] = [compact_tool_payload_for_prompt(part) for part in item[:8]]
        elif key == "raw_content":
            compact[key] = truncate_to_bytes(str(item or ""), 350)
        elif isinstance(item, (dict, list)):
            compact[key] = compact_tool_payload_for_prompt(item)
        elif isinstance(item, str):
            compact[key] = truncate_to_bytes(item, 350)
        else:
            compact[key] = item
    return compact


def compact_message_for_model(message: Dict[str, Any]) -> Dict[str, Any]:
    role = message.get("role", "user")
    content = message.get("content", "")
    if role == "tool":
        try:
            payload = json.loads(content)
            content = json.dumps(compact_tool_payload_for_prompt(payload), ensure_ascii=False)
        except Exception:
            content = truncate_to_bytes(str(content or ""), MAX_TOOL_RESULT_BYTES)
        return {"role": "user", "content": "Tool result:\n" + truncate_to_bytes(content, MAX_TOOL_RESULT_BYTES)}
    if "tool_calls" in message:
        calls = []
        for call in message.get("tool_calls", []) or []:
            function = call.get("function", {})
            calls.append({"name": function.get("name", ""), "arguments": function.get("arguments", "")})
        content = f"{content}\nController tool call summary:\n{json.dumps(calls, ensure_ascii=False)}"
        return {"role": "assistant", "content": truncate_to_bytes(content, MAX_ASSISTANT_CONTEXT_BYTES)}
    if isinstance(content, str):
        if content.startswith("Compressed evidence state:"):
            content = truncate_to_bytes(content, MAX_STATE_MESSAGE_BYTES)
        elif role == "assistant":
            content = truncate_to_bytes(content, MAX_ASSISTANT_CONTEXT_BYTES)
        else:
            content = truncate_to_bytes(content, 3000)
    return {"role": role, "content": content}


def total_message_bytes(messages: List[Dict[str, Any]]) -> int:
    return len(json.dumps(messages, ensure_ascii=False).encode("utf-8"))


def compact_messages_for_model(messages: List[Dict[str, Any]], byte_budget: int = MODEL_INPUT_BYTE_BUDGET) -> List[Dict[str, Any]]:
    compacted = [compact_message_for_model(message) for message in messages]
    while len(compacted) > MAX_MODEL_MESSAGES:
        compacted.pop(3)
    while total_message_bytes(compacted) > byte_budget and len(compacted) > 5:
        compacted.pop(3)
    if total_message_bytes(compacted) > byte_budget and compacted:
        per_message = max(500, byte_budget // max(len(compacted), 1))
        for index, message in enumerate(compacted):
            if index == 0:
                limit = min(3500, per_message)
            elif index == 1:
                limit = min(5000, per_message)
            else:
                limit = per_message
            message["content"] = truncate_to_bytes(str(message.get("content", "")), limit)
    while total_message_bytes(compacted) > byte_budget and len(compacted) > 3:
        compacted.pop(-2)
    return compacted


def make_state_message(state: Dict[str, Any], max_bytes: int = MAX_STATE_MESSAGE_BYTES) -> Dict[str, Any]:
    content = "Compressed evidence state:\n" + json.dumps(public_state_summary(state), ensure_ascii=False, indent=2)
    return {"role": "user", "content": truncate_to_bytes(content, max_bytes)}



def public_state_summary(state: Dict[str, Any]) -> Dict[str, Any]:
    refresh_candidate_span_table(state)
    return {
        "mode": state["mode"],
        "parsed_question": state.get("parsed_question", {}),
        "search_queries": state["search_queries"][-12:],
        "seen_docids": state["seen_docids"][-30:],
        "opened_docids": state["opened_docids"][-14:],
        "opened_windows": state.get("opened_windows", [])[-14:],
        "ranked_results": compact_result_summary(state["ranked_results"], max_items=8),
        "evidence_table": state["evidence_table"][-8:],
        "evidence_bank": rank_evidence_bank(state, limit=12),
        "candidate_span_table": state.get("candidate_span_table", [])[:12],
        "candidate_answers": state["candidate_answers"],
        "focused_rescue_plan": state.get("focused_rescue_plan", {}),
        "open_track": {
            "decomposition": state.get("decomposition", {}),
            "verification": state.get("verification", {}),
            "search": state.get("open_track_search", {}),
            "trace": state.get("open_track_trace", [])[-8:],
        },
        "decision": state["decision"],
    }


def evidence_constraint_hits(text: str, parsed: Dict[str, Any]) -> List[str]:
    lowered = str(text or "").lower()
    hits = []
    for term in parsed.get("hard_constraints", []) + parsed.get("answer_type_terms", []):
        term_text = str(term or "").strip()
        if term_text and term_text.lower() in lowered and term_text not in hits:
            hits.append(term_text)
    return hits


def add_evidence_item(
    state: Dict[str, Any],
    *,
    source: str,
    docid: str,
    url: str = "",
    start: int = 0,
    end: int = 0,
    text: str = "",
    round_id: int = 0,
    score: float = 0.0,
    matched_keywords: Optional[List[str]] = None,
) -> None:
    text = str(text or "").strip()
    if not text:
        return
    parsed = state.get("parsed_question") or parse_question_requirements(state.get("question", ""))
    matched = matched_keywords or evidence_constraint_hits(text, parsed)
    evidence = {
        "source": source,
        "docid": str(docid or ""),
        "url": str(url or ""),
        "start": int(start or 0),
        "end": int(end or 0),
        "round_id": int(round_id or 0),
        "text": truncate_to_bytes(text, 1800),
        "matched_keywords": matched[:10],
        "score": float(score or 0.0),
    }
    dedupe_key = (evidence["source"], evidence["docid"], evidence["start"], evidence["text"][:120])
    bank = state.setdefault("evidence_bank", [])
    existing = {
        (item.get("source"), item.get("docid"), item.get("start"), str(item.get("text", ""))[:120])
        for item in bank
    }
    if dedupe_key not in existing:
        bank.append(evidence)


def update_evidence_bank_from_execution(state: Dict[str, Any], executed: Dict[str, Any], round_id: int) -> None:
    if not executed.get("ok"):
        return
    tool_name = executed.get("tool_name", "")
    result = executed.get("tool_result")
    if tool_name == "search_many" and isinstance(result, list):
        for item in result[:30]:
            add_evidence_item(
                state,
                source="search_result",
                docid=str(item.get("docid", "")),
                url=item.get("url", ""),
                start=0,
                end=0,
                text=item.get("snippet", ""),
                round_id=round_id,
                score=float(item.get("score", 0.0) or 0.0),
            )
    elif tool_name in {"get_document", "get_document_window"} and isinstance(result, dict):
        text = result.get("snippet") or result.get("text") or ""
        add_evidence_item(
            state,
            source=tool_name,
            docid=str(result.get("docid", "")),
            url=result.get("url", ""),
            start=int(result.get("start", 0) or 0),
            end=int(result.get("end", 0) or 0),
            text=text,
            round_id=round_id,
            score=2.0,
        )
    elif tool_name == "find_in_document" and isinstance(result, dict):
        for match in result.get("matches", []) or []:
            add_evidence_item(
                state,
                source="find_in_document",
                docid=str(result.get("docid", "")),
                url=result.get("url", ""),
                start=int(match.get("start", 0) or 0),
                end=int(match.get("start", 0) or 0) + len(str(match.get("snippet", ""))),
                text=match.get("snippet", ""),
                round_id=round_id,
                score=3.0,
                matched_keywords=[str(result.get("keyword", ""))],
            )
    elif tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        for snippet in result.get("snippets", []) or []:
            add_evidence_item(
                state,
                source="collect_evidence_snippets",
                docid=str(snippet.get("docid", "")),
                url=snippet.get("url", ""),
                start=int(snippet.get("start", 0) or 0),
                end=int(snippet.get("start", 0) or 0) + len(str(snippet.get("snippet", ""))),
                text=snippet.get("snippet", ""),
                round_id=round_id,
                score=3.5,
                matched_keywords=[str(snippet.get("keyword", ""))],
            )


def score_evidence_item(item: Dict[str, Any], parsed: Dict[str, Any]) -> float:
    text = str(item.get("text", ""))
    lowered = text.lower()
    hits = evidence_constraint_hits(text, parsed)
    expected = parsed.get("expected_answer_type", "short answer")
    score = float(item.get("score", 0.0) or 0.0) + 2.5 * len(set(h.lower() for h in hits))
    source = item.get("source", "")
    if source in {"collect_evidence_snippets", "find_in_document"}:
        score += 2.0
    elif source == "get_document_window":
        score += 1.2
    if expected == "person" and re.search(r"\b[A-Z][a-z]+(?:\s+[A-Z]\.)?\s+[A-Z][a-z]+\b", text):
        score += 2.0
    elif expected == "title" and re.search(r"(?im)^\s*(?:title|chapter)\s*[:\-]", text):
        score += 2.0
    elif expected == "percentage or number" and re.search(r"\b\d+(?:\.\d+)?\s?%|\b\d+(?:\.\d+)?\s*(?:cm|centimetres|million|billion)\b", lowered):
        score += 2.0
    elif expected == "organization" and re.search(r"\b(?:Inc\.?|Corporation|Corp\.?|Company|University|Ministry|Ltd\.?)\b", text):
        score += 2.0
    elif expected == "date or year" and re.search(r"\b(?:1[5-9]\d{2}|20\d{2})\b", text):
        score += 1.5
    return score


def rank_evidence_bank(state: Dict[str, Any], limit: int = 16) -> List[Dict[str, Any]]:
    parsed = state.get("parsed_question") or parse_question_requirements(state.get("question", ""))
    ranked = []
    for item in state.get("evidence_bank", []):
        enriched = dict(item)
        enriched["score"] = score_evidence_item(enriched, parsed)
        ranked.append(enriched)
    ranked.sort(key=lambda item: (float(item.get("score", 0.0)), len(item.get("matched_keywords", []))), reverse=True)
    return ranked[:limit]


def clean_candidate_span(span: str) -> str:
    span = re.sub(r"\s+", " ", str(span or "")).strip(" \t\n\r\"'`*_.,;:")
    if not span or len(span) > 140:
        return ""
    lower = span.lower()
    if lower in STOPWORDS or lower.startswith(("http", "document", "snippet", "chapter:")):
        return ""
    if is_malformed_answer(span):
        return ""
    return span


def extract_candidate_spans_from_text(text: str, expected_type: str) -> List[str]:
    text = str(text or "")
    candidates: List[str] = []

    def add_all(pattern: str, flags: int = 0) -> None:
        for match in re.finditer(pattern, text, flags):
            span = match.group(1) if match.groups() else match.group(0)
            cleaned = clean_candidate_span(span)
            if cleaned:
                candidates.append(cleaned)

    if expected_type == "person":
        add_all(r"\b[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,3}\b")
    elif expected_type == "title":
        add_all(r"(?im)^\s*(?:title|chapter|book|article|paper)\s*[:\-]\s*([^\n]{3,120})")
        add_all(r"[\"“]([^\"”]{3,120})[\"”]")
        add_all(r"\b(?:[A-Z][A-Za-z0-9'&,-]+(?:\s+|$)){3,12}")
    elif expected_type == "organization":
        add_all(r"\b[A-Z][A-Za-z&.,'\-]+(?:\s+[A-Z][A-Za-z&.,'\-]+){0,7}\s+(?:Inc\.?|Corporation|Corp\.?|Company|University|Ministry|Agency|Ltd\.?|LLC|PLC)\b")
    elif expected_type == "percentage or number":
        add_all(r"\b\d+(?:\.\d+)?\s?%")
        add_all(r"\b\d+(?:\.\d+)?\s*(?:cm|centimetres|centimeters|million|billion|thousand|years?|miles?|km)\b", flags=re.IGNORECASE)
    elif expected_type == "date or year":
        add_all(r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}\b")
        add_all(r"\b(?:1[5-9]\d{2}|20\d{2})\b")
    elif expected_type == "place":
        add_all(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,4}(?:,\s+[A-Z][a-z]+)?\b")
    else:
        add_all(r"[\"“]([^\"”]{3,100})[\"”]")
        add_all(r"\b[A-Z][A-Za-z&.,'\-]+(?:\s+[A-Z][A-Za-z&.,'\-]+){1,6}\b")
        add_all(r"\b\d+(?:\.\d+)?\s?%")

    seen = set()
    unique = []
    for candidate in candidates:
        key = canonical_text(candidate)
        if key and key not in seen:
            seen.add(key)
            unique.append(candidate)
    return unique[:30]


def candidate_type_bonus(answer: str, expected_type: str) -> float:
    if expected_type == "person" and re.fullmatch(r"[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,3}", answer):
        return 3.0
    if expected_type == "title" and 2 <= len(answer.split()) <= 16:
        return 2.5
    if expected_type == "organization" and re.search(r"\b(Inc|Corporation|Corp|Company|University|Ministry|Agency|Ltd|LLC|PLC)\b", answer):
        return 3.0
    if expected_type == "percentage or number" and re.search(r"\d", answer):
        return 3.0
    if expected_type == "date or year" and re.search(r"\b(?:1[5-9]\d{2}|20\d{2})\b", answer):
        return 2.5
    if expected_type == "place" and re.match(r"[A-Z][a-z]+", answer):
        return 1.5
    return 0.5


def build_candidate_span_table(state: Dict[str, Any], limit: int = 20) -> List[Dict[str, Any]]:
    parsed = state.get("parsed_question") or parse_question_requirements(state.get("question", ""))
    expected = parsed.get("expected_answer_type", "short answer")
    question_key = canonical_text(state.get("question", ""))
    buckets: Dict[str, Dict[str, Any]] = {}
    for evidence in rank_evidence_bank(state, limit=24):
        for span in extract_candidate_spans_from_text(evidence.get("text", ""), expected):
            key = canonical_text(span)
            if not key or len(key) < 2:
                continue
            base = float(evidence.get("score", 0.0)) + candidate_type_bonus(span, expected)
            if key in question_key:
                base -= 2.0
            bucket = buckets.setdefault(
                key,
                {
                    "answer": span,
                    "answer_type": expected,
                    "score": 0.0,
                    "frequency": 0,
                    "docids": [],
                    "evidence_refs": [],
                    "supporting_snippets": [],
                },
            )
            bucket["score"] += max(base, 0.1)
            bucket["frequency"] += 1
            if evidence.get("docid") and evidence.get("docid") not in bucket["docids"]:
                bucket["docids"].append(evidence.get("docid"))
            bucket["evidence_refs"].append({"docid": evidence.get("docid"), "start": evidence.get("start"), "source": evidence.get("source")})
            if len(bucket["supporting_snippets"]) < 3:
                bucket["supporting_snippets"].append(truncate_to_bytes(evidence.get("text", ""), 500))
    table = list(buckets.values())
    for item in table:
        item["score"] = round(float(item["score"]) + min(item["frequency"], 4), 3)
    table.sort(key=lambda item: (item["score"], item["frequency"], len(item["docids"])), reverse=True)
    return table[:limit]


def refresh_candidate_span_table(state: Dict[str, Any]) -> List[Dict[str, Any]]:
    table = build_candidate_span_table(state)
    state["candidate_span_table"] = table
    return table


def make_evidence_packet(question: str, state: Dict[str, Any], byte_budget: int = 18000) -> Dict[str, Any]:
    parsed = state.get("parsed_question") or parse_question_requirements(question)
    state["parsed_question"] = parsed
    candidates = refresh_candidate_span_table(state)
    evidence = rank_evidence_bank(state, limit=16)
    packet = {
        "question": question,
        "parsed_question": parsed,
        "top_evidence": evidence,
        "candidate_spans": candidates,
        "opened_windows": state.get("opened_windows", [])[-14:],
        "search_queries": state.get("search_queries", [])[-12:],
    }
    while len(json.dumps(packet, ensure_ascii=False).encode("utf-8")) > byte_budget and packet["top_evidence"]:
        packet["top_evidence"].pop()
    while len(json.dumps(packet, ensure_ascii=False).encode("utf-8")) > byte_budget and packet["candidate_spans"]:
        packet["candidate_spans"].pop()
    return packet


def score_snippet_for_opening(item: Dict[str, Any], state: Dict[str, Any]) -> float:
    parsed = state.get("parsed_question") or parse_question_requirements(state.get("question", ""))
    text = str(item.get("snippet", ""))
    score = 2.5 * len(evidence_constraint_hits(text, parsed))
    keyword = str(item.get("keyword", ""))
    if keyword and keyword.lower() in text.lower():
        score += 2.0
    if item.get("start", 0):
        score += 0.5
    return score


## 5. Compact Path 与 Expansion Path

主干仍是 compact -> expansion -> adjudication。v11 在 expansion 的 evidence snippets 后追加打开命中位置上下文窗口，再交给模型回答。


In [5]:

def answer_from_state(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], prompt: str, max_tokens: int = 500) -> Dict[str, Any]:
    packet = make_evidence_packet(question, state)
    user_payload = {
        "instruction": prompt,
        "evidence_packet": packet,
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False, indent=2)},
            ],
            temperature=0.0,
            max_tokens=max_tokens,
            extra_payload=qwen_extra_payload(MODEL_NAME),
        )
        content = response["choices"][0]["message"].get("content", "")
    except Exception as exc:
        content = f"Explanation: model call failed: {exc}\nEvidence: none\nExact Answer: Unable to determine\nConfidence: 0%"
    return {"content": content, "predicted_answer": extract_exact_answer(content), "confidence": extract_confidence(content)}



def run_compact_path(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], per_query_k: int = 5, auto_open_docs: int = 3) -> Dict[str, Any]:
    state["mode"] = "compact"
    parsed = state.get("parsed_question") or parse_question_requirements(question)
    queries = build_query_pack(question, parsed, max_queries=6, max_query_chars=190)
    if not queries:
        queries = [question]
    executed = append_controller_tool_call(
        messages,
        state,
        "search_many",
        {"queries": queries, "per_query_k": per_query_k},
        round_id=0,
        note="v11 compact path: broad BM25 searches from parsed query pack.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    terms = tokenize_for_score(question) + tokenize_for_score(" ".join(parsed.get("hard_constraints", []) + parsed.get("answer_type_terms", [])))
    ranked = rank_search_results(raw_results, terms=terms, limit=10)
    state["ranked_results"] = ranked
    messages.append({"role": "assistant", "content": "v11 compact ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})
    for item in ranked[:auto_open_docs]:
        docid = str(item.get("docid", ""))
        if docid:
            append_controller_tool_call(
                messages,
                state,
                "get_document_window",
                {"docid": docid, "start": 0, "window_chars": 3000},
                round_id=0,
                note="v11 compact path: inspect compact candidate.",
            )
    keywords = ", ".join(normalize_string_list(parsed.get("hard_constraints", []) + parsed.get("answer_type_terms", []), limit=16, max_chars=80))
    top_docids = [str(item.get("docid", "")) for item in ranked[:10] if item.get("docid")]
    if keywords and top_docids:
        snippet_execution = append_controller_tool_call(
            messages,
            state,
            "collect_evidence_snippets",
            {"docids": top_docids, "keywords": keywords, "window_chars": 1100, "max_snippets": 16},
            round_id=0,
            note="v11 compact path: collect evidence snippets for candidate extraction.",
        )
        open_context_windows_from_snippets(
            messages,
            state,
            snippet_execution.get("tool_result"),
            round_id=0,
            max_windows=2,
            window_chars=3200,
            note="v11 compact path: inspect high-scoring evidence-hit context window.",
        )
    refresh_candidate_span_table(state)
    candidate = answer_from_state(question, messages, state, COMPACT_ANSWER_PROMPT)
    candidate["mode"] = "compact"
    state["candidate_answers"].append({k: candidate[k] for k in ("mode", "predicted_answer", "confidence")})
    return candidate



def plan_expansion(question: str) -> Dict[str, Any]:
    parsed = parse_question_requirements(question)
    fallback = {
        "search_queries": build_query_pack(question, parsed, max_queries=8, max_query_chars=180),
        "key_phrases": parsed.get("hard_constraints", [])[:10],
        "keywords": parsed.get("hard_constraints", [])[:16],
        "answer_type": parsed.get("expected_answer_type", "short answer"),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": PLANNER_PROMPT},
                {"role": "user", "content": json.dumps({"question": question, "parsed_question": parsed}, ensure_ascii=False)},
            ],
            temperature=0.0,
            max_tokens=500,
            extra_payload=qwen_extra_payload(MODEL_NAME),
        )
        parsed_response = extract_json_object(response["choices"][0]["message"].get("content", ""))
    except Exception:
        parsed_response = None
    if not parsed_response:
        parsed_response = fallback
    planner_queries = normalize_string_list(parsed_response.get("search_queries"), limit=8, max_chars=160)
    deterministic = fallback["search_queries"]
    key_phrases = normalize_string_list(parsed_response.get("key_phrases") or fallback["key_phrases"], limit=12, max_chars=100)
    keywords = normalize_string_list(parsed_response.get("keywords") or fallback["keywords"], limit=16, max_chars=60)
    return {
        "search_queries": normalize_string_list(deterministic + planner_queries + key_phrases, limit=10, max_chars=160),
        "key_phrases": key_phrases,
        "keywords": keywords,
        "answer_type": str(parsed_response.get("answer_type", parsed.get("expected_answer_type", "short answer"))),
    }



def run_expansion_path(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], per_query_k: int = 5, auto_open_docs: int = 4) -> Dict[str, Any]:
    state["mode"] = "expanded"
    plan = plan_expansion(question)
    messages.append({"role": "assistant", "content": "v11 expansion planner:\n" + json.dumps(plan, ensure_ascii=False, indent=2)})
    executed = append_controller_tool_call(
        messages,
        state,
        "search_many",
        {"queries": plan["search_queries"], "per_query_k": per_query_k},
        round_id=1,
        note="v11 expansion path: execute additional planned searches.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    terms = tokenize_for_score(question) + tokenize_for_score(" ".join(plan.get("keywords", []) + plan.get("key_phrases", [])))
    ranked = rank_search_results(raw_results, terms=terms, limit=12)
    state["ranked_results"] = rank_search_results(state.get("ranked_results", []) + ranked, terms=terms, limit=14)
    messages.append({"role": "assistant", "content": "v11 expansion ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})
    opened = 0
    for item in ranked:
        docid = str(item.get("docid", ""))
        if docid and docid not in state["opened_docids"]:
            append_controller_tool_call(
                messages,
                state,
                "get_document_window",
                {"docid": docid, "start": 0, "window_chars": 3000},
                round_id=1,
                note="v11 expansion path: inspect additional candidate.",
            )
            opened += 1
        if opened >= auto_open_docs:
            break
    parsed = state.get("parsed_question") or parse_question_requirements(question)
    keywords = ", ".join(normalize_string_list(plan.get("keywords", []) + plan.get("key_phrases", []) + parsed.get("answer_type_terms", []), limit=18, max_chars=80))
    top_docids = [str(item.get("docid", "")) for item in state.get("ranked_results", [])[:10] if item.get("docid")]
    if keywords and top_docids:
        snippet_execution = append_controller_tool_call(
            messages,
            state,
            "collect_evidence_snippets",
            {"docids": top_docids, "keywords": keywords, "window_chars": 1100, "max_snippets": 18},
            round_id=1,
            note="v11 expansion path: collect additional evidence snippets.",
        )
        open_context_windows_from_snippets(
            messages,
            state,
            snippet_execution.get("tool_result"),
            round_id=1,
            max_windows=3,
            window_chars=3200,
            note="v11 expansion path: inspect high-scoring evidence-hit context window.",
        )
    refresh_candidate_span_table(state)
    candidate = answer_from_state(question, messages, state, COMPACT_ANSWER_PROMPT)
    candidate["mode"] = "expanded"
    candidate["plan"] = plan
    state["candidate_answers"].append({k: candidate[k] for k in ("mode", "predicted_answer", "confidence")})
    return candidate


## 6. Open Track 工具、Agent 架构与 v11 控制器

v11 保留 v9 的高阈值 verifier。Open Track searcher 在收集 decomposition evidence snippets 后，会额外打开 3 个非重复的命中位置窗口，优先补足长文档中部/后部证据。


In [6]:
def adjudicate_candidates(question: str, state: Dict[str, Any], compact: Dict[str, Any], expanded: Dict[str, Any]) -> Dict[str, Any]:
    if not should_expand(compact):
        return {**compact, "selected_mode": "compact", "decision_reason": "compact_answer_confident_enough"}
    if should_expand(compact) and not should_expand(expanded):
        return {**expanded, "selected_mode": "expanded", "decision_reason": "compact_weak_expanded_concrete"}
    if is_unable_answer(compact.get("predicted_answer", "")) and is_unable_answer(expanded.get("predicted_answer", "")):
        return {**compact, "selected_mode": "compact", "decision_reason": "both_unable_keep_conservative"}

    evidence_summary = truncate_text(json.dumps(public_state_summary(state), ensure_ascii=False, indent=2), MAX_STATE_MESSAGE_BYTES)
    adjudication_input = {
        "role": "user",
        "content": (
            f"Question:\n{question}\n\n"
            f"Compact candidate:\n{compact.get('content', '')}\n\n"
            f"Expanded candidate:\n{expanded.get('content', '')}\n\n"
            f"Evidence summary:\n{evidence_summary}"
        ),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": ADJUDICATOR_PROMPT}, adjudication_input],
            temperature=0.0,
            max_tokens=500,
        )
        content = response["choices"][0]["message"].get("content", "")
        return {
            "mode": "adjudicated",
            "selected_mode": "adjudicated",
            "decision_reason": "adjudicator_used_for_conflict_or_weak_answers",
            "content": content,
            "predicted_answer": extract_exact_answer(content),
            "confidence": extract_confidence(content),
        }
    except Exception:
        if not is_unable_answer(expanded.get("predicted_answer", "")):
            return {**expanded, "selected_mode": "expanded", "decision_reason": "adjudicator_failed_keep_expanded"}
        return {**compact, "selected_mode": "compact", "decision_reason": "adjudicator_failed_keep_compact"}



def deterministic_decomposition(question: str) -> Dict[str, Any]:
    parsed = parse_question_requirements(question)
    hard = parsed.get("hard_constraints", [])
    expected_type = parsed.get("expected_answer_type", "short answer")
    clue_text = " ".join(hard[:10]) or question
    subquestions = normalize_string_list(
        [
            f"Find documents matching these clues: {clue_text}",
            f"Find evidence for the final requested answer type: {expected_type}",
            f"Verify the final ask: {parsed.get('final_ask', '')}",
        ],
        limit=5,
        max_chars=180,
    )
    search_focus = build_query_pack(question, parsed, max_queries=5, max_query_chars=180)
    return {
        "subquestions": subquestions,
        "constraints": hard,
        "key_entities": parsed.get("distinctive_terms", [])[:10],
        "expected_answer_type": expected_type,
        "search_focus": search_focus,
    }


def decompose_question_for_open_track(question: str, state: Dict[str, Any]) -> Dict[str, Any]:
    fallback = deterministic_decomposition(question)
    planner_input = {
        "role": "user",
        "content": json.dumps(
            {
                "question": question,
                "expected_answer_type": fallback["expected_answer_type"],
                "previous_search_queries": state.get("search_queries", [])[-10:],
                "opened_docids": state.get("opened_docids", [])[-10:],
                "top_ranked_results": compact_result_summary(state.get("ranked_results", []), max_items=6),
            },
            ensure_ascii=False,
            indent=2,
        ),
    }
    source = "model"
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": DECOMPOSER_PROMPT}, planner_input],
            temperature=0.0,
            max_tokens=500,
        )
        parsed = extract_json_object(response["choices"][0]["message"].get("content", ""))
    except Exception:
        parsed = None
    if not parsed:
        parsed = fallback
        source = "deterministic_fallback"
    decomposition = {
        "subquestions": normalize_string_list(parsed.get("subquestions") or fallback["subquestions"], limit=6, max_chars=170),
        "constraints": normalize_string_list(parsed.get("constraints") or fallback["constraints"], limit=14, max_chars=90),
        "key_entities": normalize_string_list(parsed.get("key_entities") or fallback["key_entities"], limit=10, max_chars=90),
        "expected_answer_type": str(parsed.get("expected_answer_type") or parsed.get("answer_type") or fallback["expected_answer_type"]),
        "search_focus": normalize_string_list(parsed.get("search_focus") or fallback["search_focus"], limit=5, max_chars=180),
        "source": source,
    }
    state["decomposition"] = decomposition
    record_open_track_event(state, "planner_agent", "decompose_question", decomposition)
    return decomposition


def build_decomposition_queries(question: str, decomposition: Dict[str, Any], limit: int = 6) -> List[str]:
    queries: List[str] = []
    entities = decomposition.get("key_entities", [])
    constraints = decomposition.get("constraints", [])
    answer_type = str(decomposition.get("expected_answer_type", "short answer"))
    queries.extend(decomposition.get("search_focus", [])[:4])
    if entities and constraints:
        queries.append(" ".join((entities[:5] + constraints[:5])[:10]))
    if constraints:
        queries.append(" ".join(constraints[:10]))
        queries.append(f"{answer_type} {' '.join(constraints[:8])}")
    queries.extend(make_initial_search_queries(question, max_queries=3, max_query_chars=180))
    cleaned = []
    for query in normalize_string_list(queries, limit=limit + 4, max_chars=180):
        q = canonical_text(query)
        if q.startswith(("find documents matching", "identify the", "verify all hard constraints")):
            continue
        if len(tokenize_for_score(query)) < 3:
            continue
        cleaned.append(query)
        if len(cleaned) >= limit:
            break
    return cleaned or make_initial_search_queries(question, max_queries=3, max_query_chars=180) or [question]


def run_open_track_searcher_agent(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    decomposition: Dict[str, Any],
    per_query_k: int = 4,
    auto_open_docs: int = 3,
) -> Dict[str, Any]:
    state["mode"] = "open_track_search"
    planned_queries = build_decomposition_queries(question, decomposition, limit=7)
    previous_queries = {canonical_text(query) for query in state.get("search_queries", [])}
    queries = [query for query in planned_queries if canonical_text(query) not in previous_queries]
    search_state = {"queries": queries, "opened_docids": [], "top_docids": [], "keywords": []}
    state["open_track_search"] = search_state
    if not queries:
        record_open_track_event(state, "searcher_agent", "search_skipped", {"reason": "all decomposition queries duplicate previous queries"})
        return search_state

    messages.append({"role": "assistant", "content": "v11 open-track searcher plan:\n" + json.dumps({"queries": queries, "decomposition": decomposition}, ensure_ascii=False, indent=2)})
    executed = append_controller_tool_call(
        messages,
        state,
        "search_many",
        {"queries": queries, "per_query_k": per_query_k},
        round_id=2,
        note="v11 open-track searcher: execute decomposition queries.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    terms = tokenize_for_score(question) + tokenize_for_score(
        " ".join(decomposition.get("constraints", []) + decomposition.get("key_entities", []) + [decomposition.get("expected_answer_type", "")])
    )
    ranked = rank_search_results(raw_results, terms=terms, limit=10)
    state["ranked_results"] = rank_search_results(state.get("ranked_results", []) + ranked, terms=terms, limit=12)
    messages.append({"role": "assistant", "content": "v11 open-track ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})

    opened_docids: List[str] = []
    for item in ranked:
        docid = str(item.get("docid", ""))
        if docid and docid not in state["opened_docids"]:
            append_controller_tool_call(
                messages,
                state,
                "get_document_window",
                {"docid": docid, "start": 0, "window_chars": 3000},
                round_id=2,
                note="v11 open-track searcher: inspect decomposition candidate.",
            )
            opened_docids.append(docid)
        if len(opened_docids) >= auto_open_docs:
            break

    keywords = normalize_string_list(
        decomposition.get("constraints", []) + decomposition.get("key_entities", []) + tokenize_for_score(decomposition.get("expected_answer_type", "")),
        limit=14,
        max_chars=80,
    )
    top_docids = [str(item.get("docid", "")) for item in ranked[:8] if item.get("docid")]
    if keywords and top_docids:
        snippet_execution = append_controller_tool_call(
            messages,
            state,
            "collect_evidence_snippets",
            {"docids": top_docids, "keywords": ", ".join(keywords), "window_chars": 1000, "max_snippets": 12},
            round_id=2,
            note="v11 open-track searcher: collect decomposition evidence snippets.",
        )
        opened_contexts = open_context_windows_from_snippets(
            messages,
            state,
            snippet_execution.get("tool_result"),
            round_id=2,
            max_windows=2,
            window_chars=3000,
            note="v11 open-track searcher: inspect decomposition evidence-hit context window.",
        )

    search_state.update(
        {
            "opened_docids": opened_docids,
            "top_docids": top_docids,
            "keywords": keywords,
            "retrieved_count": len(raw_results),
            "context_windows_opened": len(opened_contexts) if "opened_contexts" in locals() else 0,
        }
    )
    state["open_track_search"] = search_state
    record_open_track_event(state, "searcher_agent", "decomposition_search", search_state)
    return search_state


def normalize_verification(parsed: Dict[str, Any], candidate_answer: str, raw_content: str = "") -> Dict[str, Any]:
    support_level = canonical_text(parsed.get("support_level", "unclear")) or "unclear"
    if support_level in {"not supported", "not_supported", "no support", "unsupported answer"}:
        support_level = "unsupported"
    elif "contradict" in support_level:
        support_level = "contradicted"
    elif support_level in {"directly supported", "fully supported", "support"}:
        support_level = "supported"
    supported_value = parsed.get("supported")
    if isinstance(supported_value, str):
        supported = supported_value.strip().lower() in {"true", "yes", "supported"}
    elif supported_value is None:
        supported = support_level in {"supported", "directly supported", "fully supported"}
    else:
        supported = bool(supported_value)
    verified_answer = clean_answer_value(parsed.get("answer") or candidate_answer)
    confidence = coerce_confidence(parsed.get("confidence"), default=0)
    if is_unable_answer(verified_answer) or is_malformed_answer(verified_answer):
        supported = False
    return {
        "support_level": support_level,
        "supported": supported,
        "answer": verified_answer,
        "confidence": confidence,
        "evidence": truncate_text(str(parsed.get("evidence", "")), 600),
        "missing_constraints": normalize_string_list(parsed.get("missing_constraints"), limit=8, max_chars=100),
        "raw_content": truncate_text(raw_content, 900),
    }



def verify_candidate_answer(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], candidate: Dict[str, Any]) -> Dict[str, Any]:
    candidate_answer = clean_answer_value(candidate.get("predicted_answer", ""))
    if is_unable_answer(candidate_answer):
        verification = {
            "support_level": "not_applicable",
            "supported": False,
            "answer": "Unable to determine",
            "confidence": 0,
            "evidence": "candidate is Unable to determine",
            "missing_constraints": [],
            "raw_content": "",
        }
        state["verification"] = verification
        record_open_track_event(state, "verifier_agent", "verify_claim", {"candidate_answer": candidate_answer, "verification": verification})
        return verification

    packet = make_evidence_packet(question, state)
    verifier_input = {
        "role": "user",
        "content": json.dumps(
            {
                "question": question,
                "candidate_answer": candidate_answer,
                "candidate_output": candidate.get("content", ""),
                "evidence_packet": packet,
            },
            ensure_ascii=False,
            indent=2,
        ),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": ANSWER_VERIFIER_PROMPT}, verifier_input],
            temperature=0.0,
            max_tokens=500,
            extra_payload=qwen_extra_payload(MODEL_NAME),
        )
        content = response["choices"][0]["message"].get("content", "")
        parsed = extract_json_object(content) or {}
        verification = normalize_verification(parsed, candidate_answer=candidate_answer, raw_content=content)
    except Exception as exc:
        verification = {
            "support_level": "verifier_error",
            "supported": False,
            "answer": candidate_answer,
            "confidence": 0,
            "evidence": f"verifier failed: {exc}",
            "missing_constraints": [],
            "raw_content": "",
        }
    state["verification"] = verification
    record_open_track_event(state, "verifier_agent", "verify_claim", {"candidate_answer": candidate_answer, "verification": verification})
    return verification


VERIFIED_ACCEPT_CONFIDENCE = 70
OPEN_TRACK_RESCUE_CONFIDENCE = 65


def verification_has_missing_constraints(verification: Dict[str, Any]) -> bool:
    return bool(verification.get("missing_constraints"))


def verification_accepts(verification: Dict[str, Any], min_confidence: int = VERIFIED_ACCEPT_CONFIDENCE) -> bool:
    return (
        bool(verification.get("supported"))
        and coerce_confidence(verification.get("confidence"), default=0) >= min_confidence
        and not verification_has_missing_constraints(verification)
        and not is_unable_answer(verification.get("answer", ""))
        and not is_malformed_answer(verification.get("answer", ""))
    )


def verification_is_weak_supported(verification: Optional[Dict[str, Any]]) -> bool:
    if not verification:
        return False
    return bool(verification.get("supported")) and not verification_accepts(verification)



def open_track_finalize_answer(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    compact: Dict[str, Any],
    expanded: Optional[Dict[str, Any]],
    selected: Dict[str, Any],
) -> Dict[str, Any]:
    expected_type = state.get("decomposition", {}).get("expected_answer_type") or state.get("parsed_question", {}).get("expected_answer_type") or infer_expected_answer_type(question)
    packet = make_evidence_packet(question, state)
    finalizer_input = {
        "role": "user",
        "content": json.dumps(
            {
                "question": question,
                "expected_answer_type": expected_type,
                "compact_candidate": compact.get("content", ""),
                "expanded_candidate": expanded.get("content", "") if expanded else "",
                "selected_candidate": selected.get("content", ""),
                "evidence_packet": packet,
            },
            ensure_ascii=False,
            indent=2,
        ),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": OPEN_TRACK_FINALIZER_PROMPT}, finalizer_input],
            temperature=0.0,
            max_tokens=500,
            extra_payload=qwen_extra_payload(MODEL_NAME),
        )
        content = response["choices"][0]["message"].get("content", "")
    except Exception as exc:
        content = f"Explanation: Open Track finalizer failed: {exc}\nEvidence: none\nExact Answer: Unable to determine\nConfidence: 0%"
    candidate = {
        "mode": "open_track_finalized",
        "selected_mode": "open_track_finalized",
        "decision_reason": "open_track_planner_searcher_finalizer",
        "content": content,
        "predicted_answer": extract_exact_answer(content),
        "confidence": extract_confidence(content),
        "expected_answer_type": expected_type,
    }
    state["candidate_answers"].append({k: candidate[k] for k in ("mode", "predicted_answer", "confidence")})
    record_open_track_event(state, "finalizer_agent", "finalize_answer", {k: candidate[k] for k in ("mode", "predicted_answer", "confidence", "expected_answer_type")})
    return candidate


def make_unable_selection(reason: str, source: Dict[str, Any]) -> Dict[str, Any]:
    content = f"Explanation: {reason}\nEvidence: verification did not directly support a clean answer span\nExact Answer: Unable to determine\nConfidence: 0%"
    return {
        "mode": "open_track_rejected",
        "selected_mode": "open_track_rejected",
        "decision_reason": reason,
        "content": content,
        "predicted_answer": "Unable to determine",
        "confidence": 0,
        "rejected_candidate": source.get("predicted_answer", ""),
    }


def should_attempt_open_track_search(state: Dict[str, Any], expanded: Optional[Dict[str, Any]], selected: Dict[str, Any], verification: Optional[Dict[str, Any]] = None) -> bool:
    if expanded is None:
        return False
    answer = selected.get("predicted_answer", "")
    if is_unable_answer(answer) or is_malformed_answer(answer):
        return True
    if verification and verification.get("support_level") in {"unsupported", "contradicted"}:
        return True
    if coerce_confidence(selected.get("confidence"), default=0) < 50:
        return True
    return False


def apply_open_track_agents(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    compact: Dict[str, Any],
    expanded: Optional[Dict[str, Any]],
    selected: Dict[str, Any],
) -> Dict[str, Any]:
    selected_answer = selected.get("predicted_answer", "")
    selected_verification: Optional[Dict[str, Any]] = None

    if not is_unable_answer(selected_answer):
        selected_verification = verify_candidate_answer(question, messages, state, selected)
        if verification_accepts(selected_verification):
            return {
                **selected,
                "predicted_answer": selected_verification.get("answer", selected.get("predicted_answer", "")),
                "decision_reason": selected.get("decision_reason", "") + "_verified_high_confidence",
            }

        if expanded is None:
            if verification_is_weak_supported(selected_verification):
                return make_unable_selection("open_track_rejected_weak_supported_compact", selected)
            if is_malformed_answer(selected_answer):
                return make_unable_selection("open_track_rejected_malformed_answer", selected)
            return {**selected, "decision_reason": selected.get("decision_reason", "") + "_verification_nonblocking_kept"}

    if should_attempt_open_track_search(state, expanded, selected, selected_verification):
        decomposition = decompose_question_for_open_track(question, state)
        run_open_track_searcher_agent(question, messages, state, decomposition)
        open_track_candidate = open_track_finalize_answer(question, messages, state, compact, expanded, selected)
        if not is_unable_answer(open_track_candidate.get("predicted_answer", "")):
            open_track_verification = verify_candidate_answer(question, messages, state, open_track_candidate)
            if verification_accepts(open_track_verification, min_confidence=OPEN_TRACK_RESCUE_CONFIDENCE):
                return {
                    **open_track_candidate,
                    "predicted_answer": open_track_verification.get("answer", open_track_candidate.get("predicted_answer", "")),
                    "decision_reason": "open_track_verified_rescue_high_confidence",
                }
        if selected_verification and not verification_accepts(selected_verification):
            return make_unable_selection("open_track_rejected_low_confidence_or_incomplete_concrete", selected)
        return {**selected, "decision_reason": selected.get("decision_reason", "") + "_open_track_kept_original"}

    if is_malformed_answer(selected_answer):
        return make_unable_selection("open_track_rejected_malformed_answer", selected)
    return selected


def build_trace_record(row: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    state = result["state_summary"]
    return {
        "query_id": str(row.get("query_id", "")),
        "status": result["status"],
        "predicted_answer": result["predicted_answer"],
        "decision": state.get("decision", {}),
        "candidate_answers": state.get("candidate_answers", []),
        "search_queries": state.get("search_queries", []),
        "seen_docids": state.get("seen_docids", []),
        "opened_docids": state.get("opened_docids", []),
        "opened_windows": state.get("opened_windows", []),
        "evidence_table": state.get("evidence_table", []),
        "ranked_results": state.get("ranked_results", []),
        "focused_rescue_plan": state.get("focused_rescue_plan", {}),
        "open_track": state.get("open_track", {}),
    }


def build_open_track_trace_record(row: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    state = result["state_summary"]
    open_track = state.get("open_track", {})
    return {
        "query_id": str(row.get("query_id", "")),
        "status": result["status"],
        "predicted_answer": result["predicted_answer"],
        "decision": state.get("decision", {}),
        "decomposition": open_track.get("decomposition", {}),
        "verification": open_track.get("verification", {}),
        "search": open_track.get("search", {}),
        "events": open_track.get("trace", []),
    }



def run_v11_agent(
    question: str,
    query_id: Optional[str] = None,
    compact_per_query_k: int = 5,
    compact_auto_open_docs: int = 3,
    expansion_per_query_k: int = 5,
    expansion_auto_open_docs: int = 4,
) -> Dict[str, Any]:
    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": COMPACT_ANSWER_PROMPT},
        {"role": "user", "content": question},
    ]
    state = initial_state(question)
    compact = run_compact_path(question, messages, state, per_query_k=compact_per_query_k, auto_open_docs=compact_auto_open_docs)

    expanded = None
    if should_expand(compact):
        expanded = run_expansion_path(question, messages, state, per_query_k=expansion_per_query_k, auto_open_docs=expansion_auto_open_docs)
        selected = adjudicate_candidates(question, state, compact, expanded)
    else:
        selected = {**compact, "selected_mode": "compact", "decision_reason": "compact_answer_confident_enough"}

    selected = apply_open_track_agents(question, messages, state, compact, expanded, selected)
    refresh_candidate_span_table(state)

    state["decision"] = {
        "selected_mode": selected.get("selected_mode", selected.get("mode", "unknown")),
        "decision_reason": selected.get("decision_reason", ""),
        "compact_predicted_answer": compact.get("predicted_answer", ""),
        "compact_confidence": compact.get("confidence", 0),
        "expanded_predicted_answer": expanded.get("predicted_answer", "") if expanded else None,
        "expanded_confidence": expanded.get("confidence", 0) if expanded else None,
        "open_track_event_count": len(state.get("open_track_trace", [])),
        "open_track_verification": state.get("verification", {}),
        "open_track_search": state.get("open_track_search", {}),
        "evidence_bank_count": len(state.get("evidence_bank", [])),
        "candidate_span_count": len(state.get("candidate_span_table", [])),
    }

    messages.append({"role": "assistant", "content": selected.get("content", "")})
    return {
        "query_id": query_id,
        "status": "completed_v11_evidence_bank_concrete_first",
        "predicted_answer": selected.get("predicted_answer", ""),
        "final_output": selected.get("content", ""),
        "messages": messages,
        "state_summary": public_state_summary(state),
    }


VERIFIED_ACCEPT_CONFIDENCE = 70
OPEN_TRACK_RESCUE_CONFIDENCE = 65


def verification_has_missing_constraints(verification: Dict[str, Any]) -> bool:
    missing = verification.get("missing_constraints") or []
    support_level = canonical_text(verification.get("support_level", ""))
    return len(missing) >= 3 and support_level not in {"supported", "partial"}


def decompose_question_for_open_track(question: str, state: Dict[str, Any]) -> Dict[str, Any]:
    fallback = deterministic_decomposition(question)
    planner_input = {
        "role": "user",
        "content": json.dumps(
            {
                "question": question,
                "parsed_question": state.get("parsed_question", {}),
                "previous_search_queries": state.get("search_queries", [])[-12:],
                "opened_docids": state.get("opened_docids", [])[-12:],
                "top_ranked_results": compact_result_summary(state.get("ranked_results", []), max_items=8),
                "candidate_spans": state.get("candidate_span_table", [])[:8],
            },
            ensure_ascii=False,
            indent=2,
        ),
    }
    source = "model"
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": DECOMPOSER_PROMPT}, planner_input],
            temperature=0.0,
            max_tokens=500,
            extra_payload=qwen_extra_payload(MODEL_NAME),
        )
        parsed = extract_json_object(response["choices"][0]["message"].get("content", ""))
    except Exception:
        parsed = None
    if not parsed:
        parsed = fallback
        source = "deterministic_fallback"
    decomposition = {
        "subquestions": normalize_string_list(parsed.get("subquestions") or fallback["subquestions"], limit=6, max_chars=180),
        "constraints": normalize_string_list(parsed.get("constraints") or fallback["constraints"], limit=18, max_chars=90),
        "key_entities": normalize_string_list(parsed.get("key_entities") or fallback["key_entities"], limit=12, max_chars=90),
        "expected_answer_type": str(parsed.get("expected_answer_type") or parsed.get("answer_type") or fallback["expected_answer_type"]),
        "search_focus": normalize_string_list(parsed.get("search_focus") or fallback["search_focus"], limit=7, max_chars=180),
        "source": source,
    }
    state["decomposition"] = decomposition
    record_open_track_event(state, "planner_agent", "decompose_question", decomposition)
    return decomposition


def fallback_threshold_for_type(expected_type: str) -> float:
    return {
        "person": 8.0,
        "title": 7.5,
        "organization": 8.0,
        "percentage or number": 6.5,
        "date or year": 6.5,
        "place": 7.0,
    }.get(expected_type, 7.0)


def choose_best_span_fallback(question: str, state: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    table = refresh_candidate_span_table(state)
    if not table:
        return None
    expected = state.get("parsed_question", {}).get("expected_answer_type", "short answer")
    best = table[0]
    if float(best.get("score", 0.0)) < fallback_threshold_for_type(expected):
        return None
    evidence = " | ".join(best.get("supporting_snippets", [])[:2])
    answer = clean_answer_value(best.get("answer", ""))
    if is_unable_answer(answer) or is_malformed_answer(answer):
        return None
    confidence = max(45, min(72, int(float(best.get("score", 0.0)) * 6)))
    content = (
        "Explanation: The model paths did not produce a reliable final answer, but the evidence bank contains a high-scoring candidate span "
        f"matching the expected answer type ({expected}).\n"
        f"Evidence: {truncate_to_bytes(evidence, 900)}\n"
        f"Exact Answer: {answer}\n"
        f"Confidence: {confidence}%"
    )
    return {
        "mode": "evidence_span_fallback",
        "selected_mode": "evidence_span_fallback",
        "decision_reason": "model_unable_but_high_scoring_evidence_span",
        "content": content,
        "predicted_answer": answer,
        "confidence": confidence,
        "candidate_span": best,
    }


def apply_open_track_agents(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    compact: Dict[str, Any],
    expanded: Optional[Dict[str, Any]],
    selected: Dict[str, Any],
) -> Dict[str, Any]:
    selected_answer = selected.get("predicted_answer", "")
    selected_verification: Optional[Dict[str, Any]] = None

    if not is_unable_answer(selected_answer):
        selected_verification = verify_candidate_answer(question, messages, state, selected)
        if verification_accepts(selected_verification):
            return {
                **selected,
                "predicted_answer": selected_verification.get("answer", selected.get("predicted_answer", "")),
                "decision_reason": selected.get("decision_reason", "") + "_verified",
            }
        if selected_verification.get("support_level") in {"supported", "partial"} and not is_malformed_answer(selected_answer):
            return {**selected, "decision_reason": selected.get("decision_reason", "") + "_kept_partial_support"}

    if should_attempt_open_track_search(state, expanded, selected, selected_verification):
        decomposition = decompose_question_for_open_track(question, state)
        run_open_track_searcher_agent(question, messages, state, decomposition)
        refresh_candidate_span_table(state)
        open_track_candidate = open_track_finalize_answer(question, messages, state, compact, expanded, selected)
        if not is_unable_answer(open_track_candidate.get("predicted_answer", "")):
            open_track_verification = verify_candidate_answer(question, messages, state, open_track_candidate)
            if verification_accepts(open_track_verification, min_confidence=OPEN_TRACK_RESCUE_CONFIDENCE):
                return {
                    **open_track_candidate,
                    "predicted_answer": open_track_verification.get("answer", open_track_candidate.get("predicted_answer", "")),
                    "decision_reason": "open_track_verified_rescue",
                }
            if open_track_verification.get("support_level") in {"supported", "partial"}:
                return {**open_track_candidate, "decision_reason": "open_track_kept_partial_support"}

    fallback = choose_best_span_fallback(question, state)
    if fallback:
        state["candidate_answers"].append({k: fallback[k] for k in ("mode", "predicted_answer", "confidence")})
        record_open_track_event(state, "fallback_agent", "evidence_span_fallback", {"answer": fallback["predicted_answer"], "confidence": fallback["confidence"]})
        return fallback

    if not is_unable_answer(selected_answer) and not is_malformed_answer(selected_answer):
        return {**selected, "decision_reason": selected.get("decision_reason", "") + "_kept_concrete_after_rescue"}
    return selected


def build_trace_record(row: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    state = result["state_summary"]
    return {
        "query_id": str(row.get("query_id", "")),
        "status": result["status"],
        "predicted_answer": result["predicted_answer"],
        "decision": state.get("decision", {}),
        "parsed_question": state.get("parsed_question", {}),
        "candidate_answers": state.get("candidate_answers", []),
        "candidate_span_table": state.get("candidate_span_table", []),
        "search_queries": state.get("search_queries", []),
        "seen_docids": state.get("seen_docids", []),
        "opened_docids": state.get("opened_docids", []),
        "opened_windows": state.get("opened_windows", []),
        "evidence_table": state.get("evidence_table", []),
        "evidence_bank": state.get("evidence_bank", []),
        "ranked_results": state.get("ranked_results", []),
        "focused_rescue_plan": state.get("focused_rescue_plan", {}),
        "open_track": state.get("open_track", {}),
    }


def build_open_track_trace_record(row: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    state = result["state_summary"]
    open_track = state.get("open_track", {})
    return {
        "query_id": str(row.get("query_id", "")),
        "status": result["status"],
        "predicted_answer": result["predicted_answer"],
        "decision": state.get("decision", {}),
        "parsed_question": state.get("parsed_question", {}),
        "candidate_span_table": state.get("candidate_span_table", []),
        "decomposition": open_track.get("decomposition", {}),
        "verification": open_track.get("verification", {}),
        "search": open_track.get("search", {}),
        "events": open_track.get("trace", []),
    }


## 7. Submission、Trace 与评测

`generate_submission()` 同时写 submission、主 trace 和 Open Track 组件 trace。v11 的 trace 新增 `opened_windows`，用于判断非开头窗口是否真的被打开。


In [7]:
def build_submission_record(row: Dict[str, Any], **agent_kwargs: Any) -> tuple[Dict[str, Any], Dict[str, Any], Dict[str, Any]]:
    result = run_v11_agent(question=row["query"], query_id=str(row.get("query_id", "")), **agent_kwargs)
    record = {
        "query_id": str(row.get("query_id", "")),
        "predicted_answer": result["predicted_answer"],
        "status": result["status"],
        "messages": result["messages"],
        "state_summary": result["state_summary"],
    }
    trace = build_trace_record(row, result)
    open_track_trace = build_open_track_trace_record(row, result)
    return record, trace, open_track_trace


def generate_submission(
    dataset_path: str = HARD50_PATH,
    output_path: str = SUBMISSION_PATH,
    trace_path: str = TRACE_PATH,
    open_track_trace_path: str = OPEN_TRACK_TRACE_PATH,
    limit: Optional[int] = 50,
    **agent_kwargs: Any,
) -> List[Dict[str, Any]]:
    rows = load_jsonl(dataset_path, limit=limit)
    output_file = Path(output_path)
    trace_file = Path(trace_path)
    open_track_trace_file = Path(open_track_trace_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    trace_file.parent.mkdir(parents=True, exist_ok=True)
    open_track_trace_file.parent.mkdir(parents=True, exist_ok=True)
    records = []
    with output_file.open("w", encoding="utf-8") as fout, trace_file.open("w", encoding="utf-8") as trout, open_track_trace_file.open("w", encoding="utf-8") as otout:
        for index, row in enumerate(rows, start=1):
            print(f"[{index}/{len(rows)}] query_id={row.get('query_id', '')}")
            record, trace, open_track_trace = build_submission_record(row, **agent_kwargs)
            records.append(record)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            trout.write(json.dumps(trace, ensure_ascii=False) + "\n")
            otout.write(json.dumps(open_track_trace, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to {output_path}")
    print(f"Saved trace records to {trace_path}")
    print(f"Saved Open Track trace records to {open_track_trace_path}")
    return records


def evaluate_submission(
    submission_path: str = SUBMISSION_PATH,
    dataset_path: str = HARD50_PATH,
    output_path: str = EVAL_OUTPUT_PATH,
):
    return run_evaluation(
        submission_path=submission_path,
        dataset_path=dataset_path,
        model_name=MODEL_NAME,
        base_url=VLLM_BASE_URL,
        api_key=API_KEY,
        output_path=output_path,
        max_tokens=1024,
        verbose=True,
    )


## 8. 服务器运行入口

本地不要执行。到服务器确认 vLLM 和 BM25 索引已就绪后，再取消注释运行。

```python
# records = generate_submission(limit=50)
# summary, details = evaluate_submission()
```


In [ ]:
# Server-side entry point. Keep commented locally.
records = generate_submission(limit=50)
summary, details = evaluate_submission()


[1/50] query_id=442
[2/50] query_id=549
[3/50] query_id=26
[4/50] query_id=471
[5/50] query_id=761
[6/50] query_id=815


In [11]:
if "summary" in globals() and "details" in globals():
    print(f"\n{'='*50}")
    print("Evaluation summary")
    print(f"{'='*50}")
    print(f"total_queries:      {summary['total_queries']}")
    print(f"correct:            {summary['correct']}")
    print(f"incorrect:          {summary['incorrect']}")
    print(f"accuracy:           {summary['accuracy']:.2%}")
    print(f"avg_tool_calls:     {summary['avg_tool_calls_per_query']}")
    print(f"avg_retrieved_docs: {summary['avg_retrieved_docs_per_query']}")
    print(f"eval_model:         {summary['eval_model']}")

    print(f"\n{'='*50}")
    print("Incorrect examples (first 5)")
    print(f"{'='*50}")
    error_cases = [d for d in details if d["eval_judgment"] == "INCORRECT"]
    for case in error_cases[:5]:
        print(f"\nquery_id: {case['query_id']}")
        print(f"  Gold:      {case['gold_answer'][:100]}")
        print(f"  Predicted: {case['predicted_answer'][:100]}")
        print(f"  Reasoning: {case['eval_reasoning'][:150]}")
else:
    print("Run generate_submission() and evaluate_submission() on the server before printing the summary.")



Evaluation summary
total_queries:      50
correct:            2
incorrect:          48
accuracy:           4.00%
avg_tool_calls:     18.78
avg_retrieved_docs: 63.7
eval_model:         qwen_auto

Incorrect examples (first 5)

query_id: 442
  Gold:      THE DAWN OF AUSTRALIAN COLONISATION
  Predicted: EARLY EXPLORERS IN AUSTRALIA
  Reasoning: The predicted answer "EARLY EXPLORERS IN AUSTRALIA" is thematically related but not semantically equivalent to the gold answer "THE DAWN OF AUSTRALIAN

query_id: 549
  Gold:      Marguerite Smith
  Predicted: Belle da Costa Greene
  Reasoning: The predicted answer "Belle da Costa Greene" is not semantically equivalent to the gold answer "Marguerite Smith" and does not match the specific crit

query_id: 26
  Gold:      Binochios
  Predicted: United States
  Reasoning: The predicted answer "United States" is not semantically equivalent to the gold answer "Binochios" and does not address the specific question about th

query_id: 471
  Gold:      Scott